# 可变形注意力

> 目标：把“标准注意力为什么贵、可变形注意力到底改了什么、输入里每个元素代表什么、代码里每一步在做什么”串成一条连续逻辑。

> 这份笔记默认你已经见过 Transformer，但不假设你已经理解 Deformable DETR 或 BEVFormer。

> 阅读路线：
1. 先回顾标准注意力里的 query、key、value 到底是什么意思
2. 再看视觉任务里为什么不想让每个 query 和整张特征图全量匹配
3. 然后引入卷积与可变形卷积，理解“规则采样”到“可学习采样”的变化
4. 最后再看可变形注意力：参考点、偏移量、采样权重分别承担什么角色
5. 对照 PyTorch 代码，看这些概念如何落到张量实现上

> 一句话先说结论：
- 标准注意力是“先和所有位置算相关性，再决定关注谁”。
- 可变形注意力是“直接预测应该去哪些位置采样，再把采样结果加权融合”。

> 因此它不是在标准注意力上做一个小修小补，而是把“全量匹配”改成了“稀疏、连续、可学习采样”。

## 1. 可变形注意力和可变形卷积的关系

两者的共同思想都是：
- 不在固定规则位置取样。
- 让网络动态决定去哪里取信息。

但两者也有明显区别：
- 可变形卷积通常是围绕局部邻域做可学习采样。
- 可变形注意力则是围绕一个参考点，在更大范围内做稀疏采样与加权融合。

你可以先把可变形注意力理解成：
- 保留了“动态采样位置”这个思想。
- 同时引入了注意力里的“内容相关加权融合”能力。

## 2. 从标准注意力一步步走到可变形注意力

> 这一节只做一件事：把“标准注意力”里的元素，平滑地映射到“可变形注意力”里。

### 2.1 标准注意力里，query、key、value 分别是什么

设有一组输入特征 $x$，经过线性映射后得到：
- $Q = xW_Q$
- $K = xW_K$
- $V = xW_V$

对某个 query $q_i$，标准注意力做的是：

$$
\mathrm{Attn}(q_i) = \sum_j \alpha_{ij} v_j, \qquad
\alpha_{ij} = \mathrm{softmax}\left( \frac{q_i k_j^T}{\sqrt{d}} \right)
$$

这里每个元素的含义是：
- `query`：当前我想更新的那个位置或 token。
- `key`：候选信息位置的“索引签名”，用来和 query 算相似度。
- `value`：真正要被取回并聚合的信息。

所以标准注意力本质上是：
- 先拿 query 和所有 key 做匹配。
- 再用匹配分数去加权所有 value。

### 2.2 放到视觉特征图上，会发生什么

如果把一张特征图展平成 $N=H \times W$ 个位置：
- 每个空间位置都可以看成一个 token。
- 每个 query 可能都要和全部 $N$ 个 key 交互。
- 如果 query 数也很多，那么相关性矩阵就会很大。

这就是视觉里标准全局注意力昂贵的根源：
- 不是聚合本身太难。
- 而是“先和所有位置比较一遍”这一步太贵。

### 2.3 一个关键想法：真的需要和所有 key 比吗

在很多视觉任务中，答案通常是否定的。
- 一个目标只和图像中少量区域强相关。
- 一个 BEV 网格点也只会对应少量视角中的局部区域。
- 大量无关位置其实只是陪跑。

于是就有一个自然问题：
- 能不能不要先和所有 key 做匹配？
- 能不能直接预测“应该去哪里看”？

可变形注意力给出的答案就是：可以。

### 2.4 从“全量匹配”变成“稀疏采样”

标准注意力的流程是：
1. 对全部 key 位置做相似度计算。
2. 用 softmax 得到全局权重。
3. 对全部 value 做加权和。

可变形注意力把它改成：
1. 先给每个 query 一个参考点 `reference_point`。
2. 再预测少量采样偏移 `offsets`。
3. 在这些采样位置上，从 value 特征图中取值。
4. 对这些采样值做加权求和。

这意味着：
- 不再显式构造“query 对所有 key”的大矩阵。
- key 不再以“离散候选集合”的方式出现。
- 取而代之的是“连续坐标上的采样位置”。

### 2.5 那 key 去哪了

这是很多人第一次看可变形注意力时最困惑的点。

一个容易理解的说法是：
- 在标准注意力里，key 的职责是“告诉 query 哪些位置相关”。
- 在可变形注意力里，这个职责被“参考点 + 偏移量 + 采样权重”部分接管了。

也就是说，在很多实现里：
- 不再显式地构造一整张 `K` 去和 `Q` 做全量点积。
- 而是由 query 直接预测出该去哪里采样，以及每个采样点该占多大权重。

所以你可以把可变形注意力理解为：
- `query` 仍然存在，而且是核心驱动信号。
- `value` 仍然存在，因为最终被聚合的信息还是来自特征图。
- `key` 的“全局离散匹配”角色被弱化或隐式化，变成了“直接预测采样策略”。

In [1]:
import torch
import torch.nn as nn

# 一个最小标准注意力例子：1 个 query 对 4x4 特征图的全部位置打分
torch.manual_seed(0)

feature_map = torch.randn(1, 4, 4, 8)
flattened_tokens = feature_map.view(1, 16, 8)

query_token = torch.randn(1, 1, 8)
keys = flattened_tokens
values = flattened_tokens

attention_scores = torch.matmul(query_token, keys.transpose(-1, -2)) / (8 ** 0.5)
attention_weights = torch.softmax(attention_scores, dim=-1)
attention_output = torch.matmul(attention_weights, values)

print("全部候选位置数:", keys.shape[1])
print("attention_weights shape:", attention_weights.shape)
print("权重和是否为 1:", attention_weights.sum(dim=-1))
print("输出向量 shape:", attention_output.shape)
print("注意力最大的空间位置索引:", attention_weights[0, 0].argmax().item())

全部候选位置数: 16
attention_weights shape: torch.Size([1, 1, 16])
权重和是否为 1: tensor([[1.]])
输出向量 shape: torch.Size([1, 1, 8])
注意力最大的空间位置索引: 0


## 3. 输入元素到底是什么：和传统 Q、K、V 的对应关系

> 这一节专门解决“模块输入里每个张量到底代表谁”的问题。

为了便于理解，先看单尺度版本。假设我们有：
- `query`: 形状为 $(B, N_q, C)$
- `value`: 形状为 $(B, H, W, C)$
- `reference_points`: 形状为 $(B, N_q, 2)$

它们分别表示：

### 3.1 query：谁要被更新

`query` 对应标准注意力里的 `Q`。
- 它代表当前要更新的那批查询向量。
- 在检测里，它可以是 object query。
- 在 BEVFormer 里，它可以是 BEV query。
- 在更一般的场景里，它就是“我现在想从特征图里取信息来更新谁”。

所以每个 query 都会独立地产生：
- 一个或多个参考点
- 若干个采样偏移
- 若干个采样权重

### 3.2 value：真正被读取的信息源

`value` 对应标准注意力里的 `V`。
- 它通常来自某张特征图或多层特征图。
- 真正被双线性采样、再被加权融合的内容，就是它。

注意这里 `value` 不是展平的 token 序列，而是保留了二维空间结构的特征图。
这很重要，因为：
- 可变形注意力要在连续坐标上采样。
- 所以保留 $(H, W)$ 的几何布局会更自然。

### 3.3 reference_points：粗定位锚点

`reference_points` 不是标准注意力里的原生元素，它是可变形注意力新增的关键变量。
- 它表示这个 query 大概应该去哪里找信息。
- 可以理解成“粗定位”。
- 后续所有偏移采样，都是围绕它展开。

它的来源依任务而不同：
- 在 Deformable DETR 中，常由 query 经过线性层预测并归一化。
- 在 BEVFormer 中，常与 3D 到 2D 的几何投影有关。

### 3.4 sampling_offsets：细粒度搜索方向

`sampling_offsets` 通常由 query 通过线性层预测得到。
- 形状常写成 $(B, N_q, N_h, K, 2)$。
- 表示每个 query、每个 head、每个采样点的二维偏移。

它的作用相当于：
- 参考点先告诉你“中心大概在哪”。
- 偏移量再告诉你“具体往周围哪些位置看”。

### 3.5 attention_weights：对采样结果做融合

`attention_weights` 同样通常由 query 预测得到。
- 形状常写成 $(B, N_q, N_h, K)$。
- 每个 head 的 $K$ 个采样点会经过 softmax 归一化。

它对应的是注意力里“加权融合”的那部分能力。
只是这里被加权的对象，不再是全部 value 位置，而是少量被采样出来的特征值。

### 3.6 用一句对照总结

把标准注意力和可变形注意力并排看：
- 标准注意力：`query` 与全部 `key` 匹配，再聚合全部 `value`。
- 可变形注意力：`query` 直接预测采样策略，在 `value` 上稀疏取样并聚合。

因此新增的核心对象就是：
- `reference_points`：先去哪里附近看
- `sampling_offsets`：具体采哪些点
- `attention_weights`：这些点如何融合

In [2]:
# 观察 reference_points、sampling_offsets、sampling_locations 的形状与数值关系
batch_size = 1
num_query = 2
num_heads = 2
num_points = 4

reference_points_demo = torch.tensor([[[0.50, 0.50], [0.25, 0.75]]])
sampling_offsets_demo = torch.tensor(
    [[[[[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]],
       [[2.0, 0.0], [0.0, 2.0], [-2.0, 0.0], [0.0, -2.0]]],
      [[[1.0, 1.0], [-1.0, 1.0], [-1.0, -1.0], [1.0, -1.0]],
       [[0.5, 0.5], [-0.5, 0.5], [-0.5, -0.5], [0.5, -0.5]]]]],
    dtype=torch.float32,
 )

height, width = 20, 30
normalizer = torch.tensor([width, height], dtype=torch.float32).view(1, 1, 1, 1, 2)
sampling_locations_demo = (
    reference_points_demo[:, :, None, None, :] + sampling_offsets_demo / normalizer
)

print("reference_points shape:", reference_points_demo.shape)
print("sampling_offsets shape:", sampling_offsets_demo.shape)
print("sampling_locations shape:", sampling_locations_demo.shape)
print("第 1 个 query、第 1 个 head 的采样位置:\n", sampling_locations_demo[0, 0, 0])

reference_points shape: torch.Size([1, 2, 2])
sampling_offsets shape: torch.Size([1, 2, 2, 4, 2])
sampling_locations shape: torch.Size([1, 2, 2, 4, 2])
第 1 个 query、第 1 个 head 的采样位置:
 tensor([[0.5333, 0.5000],
        [0.5000, 0.5500],
        [0.4667, 0.5000],
        [0.5000, 0.4500]])


## 4. 数学表达：从标准注意力公式改写到可变形注意力公式

> 这一步不要急着背公式，先看“公式里哪些部分被替换了”。

### 4.1 标准注意力在做什么

对第 $i$ 个 query，标准注意力是：

$$
\mathrm{Attn}(q_i) = \sum_{j=1}^{N} \alpha_{ij} v_j, \qquad
\alpha_{ij} = \mathrm{softmax}\left( \frac{q_i k_j^T}{\sqrt{d}} \right)
$$

这里有两个核心步骤：
- 先对所有 $j$ 计算相关性。
- 再对所有 $v_j$ 做加权和。

### 4.2 可变形注意力改写了哪一步

可变形注意力没有再枚举全部 $j$。
它直接把“遍历所有位置”替换成“只访问少量采样点”。

设：
- query 表示为 $q_i$，共有 $M$ 个 query。
- value 特征图为 $x \in \mathbb{R}^{H \times W \times C}$。
- 第 $i$ 个 query 的参考点为 $p_i \in [0, 1]^2$。
- 第 $m$ 个 head、第 $k$ 个采样点的偏移量为 $\Delta p_{imk}$。
- 对应权重为 $A_{imk}$。

则单尺度可变形注意力写成：

$$
\mathrm{DeformAttn}(q_i, x) = \sum_{m=1}^{N_h} W_m \left( \sum_{k=1}^{K} A_{imk} \cdot x\big(p_i + \Delta p_{imk}\big) \right)
$$

这里要重点理解三件事：
- $x(p_i + \Delta p_{imk})$ 表示通过坐标来索引特征图x
- $p_i + \Delta p_{imk}$ 不一定落在整数网格上，所以要做双线性插值。
- $A_{imk}$ 不是对全图所有位置的权重，而只是对少量采样点的权重。
- 每个 head 可以学习不同的采样模式。

### 4.3 为什么这可以看成“注意力”的一种形式

虽然没有显式写出 $QK^T$，但它仍然具备注意力的两个核心特征：
- 由 query 决定“该关注哪里”。
- 对取回的信息做内容相关的加权融合。

区别只在于：
- 标准注意力通过“和全部 key 匹配”来决定关注位置。
- 可变形注意力通过“直接预测采样位置和权重”来决定关注位置。

### 4.4 多尺度版本再多了什么

如果是多尺度版本，设第 $l$ 层特征图为 $x^l$，则进一步写成：

$$
\mathrm{MSDeformAttn}(q_i) = \sum_{m=1}^{N_h} W_m \left( \sum_{l=1}^{L} \sum_{k=1}^{K} A_{imlk} \cdot x^l\big(p_i^l + \Delta p_{imlk}\big) \right)
$$

这比单尺度只多了一个维度：
- 从一张特征图采样，变成从 $L$ 张不同分辨率特征图采样。

### 4.5 复杂度为什么更低

如果标准注意力要访问全部 $HW$ 个位置，而可变形注意力每个 head 只采样 $K$ 个点，那么：
- 标准注意力的访问规模接近 $HW$。
- 可变形注意力的访问规模接近 $N_h \times K$。

当 $K \ll HW$ 时，节省会非常明显。
这也是它特别适合高分辨率视觉特征的原因。

In [3]:
# 用一个最小例子看“全量加权”和“稀疏采样加权”的差异
value_map = torch.arange(1, 17, dtype=torch.float32).view(1, 4, 4, 1)

# 标准注意力视角：对全部 16 个位置做加权
all_values = value_map.view(1, 16, 1)
dense_weights = torch.softmax(torch.tensor([[0.2] * 16]), dim=-1).view(1, 16, 1)
dense_output = (dense_weights * all_values).sum(dim=1)

# 可变形注意力视角：只取 4 个采样点做加权
sparse_sample_indices = torch.tensor([5, 6, 9, 10])
sparse_values = all_values[:, sparse_sample_indices]
sparse_weights = torch.softmax(torch.tensor([[1.0, 2.0, 2.0, 1.0]]), dim=-1).view(1, 4, 1)
sparse_output = (sparse_weights * sparse_values).sum(dim=1)

print("标准注意力访问的位置数:", all_values.shape[1])
print("可变形注意力访问的位置数:", sparse_values.shape[1])
print("标准注意力输出:", dense_output.squeeze().item())
print("稀疏采样聚合输出:", sparse_output.squeeze().item())

标准注意力访问的位置数: 16
可变形注意力访问的位置数: 4
标准注意力输出: 8.5
稀疏采样聚合输出: 8.5


## 5. 用一个 query 的视角看一遍完整流程

> 如果你总觉得公式里变量太多，可以只盯住“一个 query”来看。

假设：
- 我们当前只看第 $i$ 个 query。
- 它对应一个向量 $q_i \in \mathbb{R}^{C}$。
- 它的参考点是 $p_i=(0.6, 0.4)$，表示大概去特征图右侧偏上区域找信息。
- 现在有 2 个 head，每个 head 采样 4 个点。

那么这个 query 的工作流程可以写成：

### 第一步：先确定大致看哪里

模型先为这个 query 给出一个参考点 $p_i$。
这个点不是最终采样点，而是一个中心锚点。

### 第二步：每个 head 进一步预测偏移

例如某个 head 预测出 4 个偏移：
- $(-0.03, 0.01)$
- $(0.02, -0.04)$
- $(0.00, 0.05)$
- $(0.04, 0.02)$

于是最终采样位置就是：
- $p_i + \Delta p_{i11}$
- $p_i + \Delta p_{i12}$
- $p_i + \Delta p_{i13}$
- $p_i + \Delta p_{i14}$

这些位置通常是浮点坐标，不会正好落在整数像素中心上。

### 第三步：在 value 特征图上取值

因为采样位置是连续坐标，所以不能直接用整数索引。
通常会用双线性插值：
- 找到附近的 4 个离散网格点
- 按距离做加权
- 得到该连续位置的特征向量

这一步就是代码里的 `grid_sample` 在做的事。

### 第四步：对采样到的若干特征做加权融合

假设这个 head 对 4 个采样点预测出的权重是：

$$
[0.1, 0.2, 0.5, 0.2]
$$

那么这个 head 的输出就是：

$$
0.1 v_1 + 0.2 v_2 + 0.5 v_3 + 0.2 v_4
$$

其中 $v_1, v_2, v_3, v_4$ 是 4 个采样位置通过双线性插值得到的特征向量。

### 第五步：多个 head 的结果再拼起来

每个 head 都会做一遍“预测偏移 -> 采样 -> 加权求和”。
最后把所有 head 的结果拼接起来，再过一个线性层，得到这个 query 的更新结果。

所以整个过程本质上是：
- query 不再对全图搜索。
- query 直接决定采样策略。
- value 特征图作为被读取的信息源。

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimplifiedDeformableAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, num_points):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError("embed_dim 必须能被 num_heads 整除")

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_points = num_points
        self.head_dim = embed_dim // num_heads

        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.offset_proj = nn.Linear(embed_dim, num_heads * num_points * 2)
        self.weight_proj = nn.Linear(embed_dim, num_heads * num_points)
        self.output_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, value, reference_points):
        """
        query: (B, Nq, C)
            要被更新的查询向量，对应标准注意力中的 Q
        value: (B, H, W, C)
            被采样和聚合的特征图，对应标准注意力中的 V
        reference_points: (B, Nq, 2)
            每个 query 的粗定位锚点，坐标范围为 [0, 1]
        """
        batch_size, num_query, _ = query.shape
        _, height, width, _ = value.shape

        # 先把 value 投影到多头注意力所需的通道空间
        # 形状: (B, H, W, C) -> (B, H, W, num_heads * head_dim) -> (B*num_heads, head_dim, H, W)
        projected_value = self.value_proj(value)
        projected_value = projected_value.view(
            batch_size, height, width, self.num_heads, self.head_dim
        )
        projected_value = projected_value.permute(0, 3, 4, 1, 2).contiguous()
        projected_value = projected_value.view(
            batch_size * self.num_heads, self.head_dim, height, width
        )

        # 由 query 直接预测采样偏移和采样权重
        # sampling_offsets 形状: (B, Nq, num_heads * num_points * 2) -> (B, Nq, num_heads, num_points, 2)
        # 这里2表示每个采样点的 (x, y) 偏移,num_points 表示每个 head 的采样点数量，这些采样点的数量也就是最终用来做加权融合的点的数量
        sampling_offsets = self.offset_proj(query)
        sampling_offsets = sampling_offsets.view(
            batch_size, num_query, self.num_heads, self.num_points, 2
        )

        # attention_weights 形状: (B, Nq, num_heads * num_points) -> (B, Nq, num_heads, num_points)
        # 为什么是这样，因为每个query 对每个 head 的 num_points 个采样点都要预测一个权重，这些权重会用来对采样到的特征做加权融合
        attention_weights = self.weight_proj(query)
        attention_weights = attention_weights.view(
            batch_size, num_query, self.num_heads, self.num_points
        )
        # 经过 softmax 归一化后，attention_weights 就可以看作是每个 query 对其采样点的注意力分布了
        attention_weights = F.softmax(attention_weights, dim=-1)

        # 把偏移量从像素尺度归一化到 [0, 1] 参考系
        normalizer = query.new_tensor([width, height])
        # 创建一个与 query 张量具有相同设备（device）和数据类型（dtype）的新张量，并用列表 [width, height] 中的值填充形状是(2,),tenser([width, height]) 
        normalizer = normalizer.view(1, 1, 1, 1, 2)
        # (batch_size, num_heads, num_queries, num_points, 2)，最后两位是 (x, y) 坐标的最大值，即宽度高度
        # (B, Nq, num_heads, num_points, 2)/(1, 1, 1, 1, 2) -> (B, Nq, num_heads, num_points, 2)，
        # 每个采样点的偏移量除以最大值都被归一化了
        sampling_locations = (
            reference_points[:, :, None, None, :] + sampling_offsets / normalizer
        )

        # grid_sample 需要 [-1, 1] 坐标，且最后一维顺序是 (x, y)
        sampling_grid = sampling_locations * 2.0 - 1.0
        # 形状： (B, Nq, num_heads, num_points, 2) -> (B, num_heads, Nq, num_points, 2) -> (B*num_heads, Nq, num_points, 2)
        sampling_grid = sampling_grid.permute(0, 2, 1, 3, 4).contiguous()
        sampling_grid = sampling_grid.view(
            batch_size * self.num_heads, num_query, self.num_points, 2
        )

        # 在连续坐标上做双线性采样
        # (B*num_heads, head_dim, H, W)，value图二维的作为 grid_sample 的查询依据
        # (B*num_heads, Nq, num_points, 2)，
        # 每个采样点的坐标是(x_norm, y_norm),-1 和 1 分别对应输入特征图最左侧像素的中心和最右侧像素的中心
        # 反归一化得到浮点数的索引，然后进行双线性插值采样，得到每个采样点的特征值
        sampled_value = F.grid_sample(
            projected_value,  #输入特征图，形状为 (N, C, H_in, W_in)
            sampling_grid,   #采样网格，形状为 (N, H_out, W_out, 2)
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )
        # 输出形状为 (N, C, H_out, W_out)
        # 采样结果形状: (B*num_heads, head_dim, Nq, num_points) -> (B, num_heads, head_dim, Nq, num_points)
        sampled_value = sampled_value.view(
            batch_size, self.num_heads, self.head_dim, num_query, self.num_points
        )
        # 调整维度顺序到 (B, num_heads, Nq, num_points, head_dim)，方便后续加权融合
        sampled_value = sampled_value.permute(0, 3, 1, 4, 2).contiguous()

        # 对每个 head 的若干采样点做加权融合
        # attention_weights 形状: (B, Nq, num_heads, num_points) -> (B, Nq, num_heads, num_points, 1)，
        attention_weights = attention_weights.unsqueeze(-1)
        # sampled_value 形状: (B, Nq, num_heads, num_points, head_dim)* (B, Nq, num_heads, num_points, 1) -> (B, Nq, num_heads, head_dim)，
        # 相当于每个 query 每个 head 的 num_points 个采样点的特征值根据 attention_weights 预测的权重做加权求和，得到每个 head 的输出特征
        output = (sampled_value * attention_weights).sum(dim=3)
        # 调整维度顺序到 (B, Nq, num_heads * head_dim) -> (B, Nq, C)
        output = output.view(batch_size, num_query, self.embed_dim)
        # 最后再经过一个线性投影，得到最终的输出特征
        output = self.output_proj(output)
        # 输出形状: (B, Nq, C)
        return output

总结起来，就是由 query 直接预测采样偏移和采样权重，value 投影到多头注意力所需的通道空间，作为被查询的二维图，query得到的偏移量加上传入的参考点得到绝对位置坐标，然后进行归一化，再使用value构建的图与query的绝对归一化坐标双线性插值，得到num_point个实际用于加权求和的value，最后有attention_weight和num_points个value进行加权得到最终输出，输出与query形状一致。

In [5]:
batch_size = 2
num_query = 6
height, width = 20, 30
embed_dim = 64
num_heads = 8
num_points = 4

torch.manual_seed(0)

module = SimplifiedDeformableAttention(
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_points=num_points,
)

query = torch.randn(batch_size, num_query, embed_dim)
value = torch.randn(batch_size, height, width, embed_dim)
reference_points = torch.rand(batch_size, num_query, 2)

output = module(query, value, reference_points)
print("query shape:", query.shape)
print("value shape:", value.shape)
print("reference_points shape:", reference_points.shape)
print("output shape:", output.shape)

query shape: torch.Size([2, 6, 64])
value shape: torch.Size([2, 20, 30, 64])
reference_points shape: torch.Size([2, 6, 2])
output shape: torch.Size([2, 6, 64])


## 6. 代码实现逐段解释：把概念和代码一一对上

> 这一版代码是教学版，不是官方高性能 CUDA 版本，但逻辑链路是完整的。

### 6.1 为什么代码里几乎看不到显式的 key

这段代码里你会发现：
- 有 `query`
- 有 `value`
- 但没有单独写出一个 `key` 张量

原因不是 key 不重要，而是这份简化实现采用了可变形注意力里更容易理解的一种视角：
- 让 `query` 直接预测采样位置和采样权重
- 不再显式构造“query 与全体 key 的相似度矩阵”

所以这里相当于把标准注意力里的：
- “通过 key 找相关位置”
改成了：
- “直接预测该采样哪些位置”

### 6.2 `value_proj` 在做什么

`value_proj` 对 `value` 做线性映射：
- 输入还是空间特征图
- 但通道会被投影到当前注意力模块所用的嵌入空间

随后代码把它 reshape 成多头形式。
这一步对应的是：
- 不同 head 各自处理一部分通道
- 每个 head 后续会学习不同的采样模式

### 6.3 `offset_proj` 和 `weight_proj` 在做什么

这两个线性层都只吃 `query`：
- `offset_proj(query)` 预测采样偏移
- `weight_proj(query)` 预测采样权重

这正是“由 query 决定采样策略”的代码体现。
也就是说，query 不再先去和所有 key 打分，而是直接输出：
- 去哪采
- 采几点
- 每个点占多少比重

### 6.4 为什么要除以 `[width, height]`

代码里有一步：

```python
normalizer = query.new_tensor([width, height]).view(1, 1, 1, 1, 2)
sampling_locations = reference_points[:, :, None, None, :] + sampling_offsets / normalizer
```

这是因为：
- `reference_points` 用的是归一化坐标 $[0,1]$
- 而偏移量通常更接近像素尺度的位移概念

所以要把偏移量也缩放到同一个参考系，再和参考点相加。

### 6.5 `grid_sample` 为什么是关键

`grid_sample` 的作用可以概括成一句话：
- 给我一组连续坐标，我帮你从二维特征图上做双线性插值取值。

这一步非常关键，因为采样点通常不是整数网格点。
如果没有这一步：
- 采样位置只能落在离散像素上
- 偏移量的连续可学习性就会大打折扣

所以从实现角度说：
- 可变形注意力能“可变形”
- 很大程度上就依赖这种连续坐标采样机制

### 6.6 最后那一步加权求和，对应的是哪部分注意力

代码最后：

```python
output = (sampled_value * attention_weights).sum(dim=3)
```

这一步就是注意力里的“聚合”。
只不过聚合对象不再是全图所有 value，而是少量采样得到的 value。

所以把它和标准注意力对照起来：
- 标准注意力：对全部 value 做加权和
- 可变形注意力：对少量采样 value 做加权和

### 6.7 这份简化代码省略了什么

为了教学清晰，这里刻意省略了一些工程细节：
- 没有实现多尺度 feature level
- 没有实现更复杂的 reference point 生成方式
- 没有使用官方 CUDA kernel 做高效采样
- 没有处理更复杂的 mask 与 padding 逻辑

但对理解原理来说，它已经保留了最核心的 4 件事：
- query 驱动采样策略
- reference point 提供粗定位
- offsets 提供细粒度采样
- sampled value 再经过 attention weights 融合

## 7. 和 BEVFormer 中用法的关系

> 当你把前面的概念吃透，再看 BEVFormer，就不会觉得“为什么又冒出参考点和采样点”。

可以这样理解：
- 每个 BEV query 对应鸟瞰图上的一个网格位置。
- 这个位置通过相机外参与内参，可以投影到各个相机视角图像上。
- 投影结果提供了一个很自然的参考点。
- 接着模型再围绕这个参考点学习少量偏移做细粒度采样。
- 来自不同相机、不同尺度的采样结果再融合回这个 BEV query。

所以 BEVFormer 中的空间交叉注意力，本质上就是：
- 几何投影先给一个粗定位。
- 可变形注意力再做内容相关的稀疏采样。
- 最后把多视角信息聚合回 BEV 空间。


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DeformableBEVFusion(nn.Module):
    """动手学深度学习风格：基于 Query 的多尺度可变形 BEV 特征融合模块
    
    设计理念：
        1. 极致的性能优化：先做 Value 投影再做多尺度池化与采样，彻底避免在采样内部做全图高维线性变换。
        2. 极度详细的维度标注：让每一个 Reshape/Permute 的物理含义都清晰可见。
    """
    def __init__(self, embed_dim=256, num_heads=8, num_levels=3, num_points=4, dropout=0.1):
        super().__init__()
        
        # 基础参数校验
        if embed_dim % num_heads != 0:
            raise ValueError(f"embed_dim ({embed_dim}) 必须能被 num_heads ({num_heads}) 整除！")

        self.embed_dim = embed_dim      # 特征总通道数 C
        self.num_heads = num_heads      # 多头注意力的头数 M
        self.num_levels = num_levels    # 多尺度金字塔层数 L
        self.num_points = num_points    # 每个头在每个尺度上的采样点数 K
        self.head_dim = embed_dim // num_heads  # 每个头的特征维度 C_head

        # -----------------------------------------------------------------
        # 线性投影层定义
        # -----------------------------------------------------------------
        # 1. 将 BEV 特征映射到 Query 的隐空间 (Value 投影)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        
        # 2. 从 Query 预测归一化的参考点中心坐标 (x, y) -> 维度: L * 2
        self.reference_points_proj = nn.Linear(embed_dim, num_levels * 2)
        
        # 3. 从 Query 预测采样点的偏移量 Offset (dx, dy) -> 维度: M * L * K * 2
        self.offset_proj = nn.Linear(embed_dim, num_heads * num_levels * num_points * 2)
        
        # 4. 从 Query 预测采样点的注意力权重 Weight -> 维度: M * L * K
        self.weight_proj = nn.Linear(embed_dim, num_heads * num_levels * num_points)
        
        # 5. 聚合后的特征输出投影层
        self.output_proj = nn.Linear(embed_dim, embed_dim)

        # 6. 全局上下文融合层 (将 Query 更新后的特征回注入 BEV 特征图)
        self.context_fusion = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim)
        )

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)

        # 参数初始化
        self._init_parameters()

    def _init_parameters(self):
        """参数初始化策略：
        对于 Offset 预测层，必须初始化为 0！这样在训练初期，采样点会默认落在 Reference Point 附近，
        避免一开始随机乱采导致梯度崩溃，这是可变形注意力能收敛的关键技巧！
        """
        nn.init.xavier_uniform_(self.value_proj.weight)
        nn.init.constant_(self.value_proj.bias, 0.0)
        
        nn.init.xavier_uniform_(self.reference_points_proj.weight)
        nn.init.constant_(self.reference_points_proj.bias, 0.0)

        # 【核心技巧】Offset 权重与偏置全部归零
        nn.init.constant_(self.offset_proj.weight, 0.0)
        nn.init.constant_(self.offset_proj.bias, 0.0)

        nn.init.xavier_uniform_(self.weight_proj.weight)
        nn.init.constant_(self.weight_proj.bias, 0.0)
        
        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.constant_(self.output_proj.bias, 0.0)

    def _build_multi_scale_features(self, bev_feature):
        """预先计算全图的 Value 投影并构建多尺度特征金字塔
        
        输入: 
            bev_feature: [B, C, H, W]
        输出: 
            feats: 列表，包含 num_levels 个特征图
                   level 0: [B, C, H, W]
                   level 1: [B, C, H/2, W/2]
                   level 2: [B, C, H/4, W/4] ...
        """
        b, c, h, w = bev_feature.shape
        
        # 1. 展平通道做全图线性映射
        # [B, C, H, W] -> [B, C, H*W] -> [B, H*W, C]
        bev_tokens = bev_feature.flatten(2).transpose(1, 2)
        
        # [B, H*W, C] -> [B, H*W, C]
        bev_proj = self.value_proj(bev_tokens)
        
        # 还原回 2D 特征图形状: [B, H*W, C] -> [B, C, H*W] -> [B, C, H, W]
        bev_proj = bev_proj.transpose(1, 2).view(b, c, h, w).contiguous()

        # 2. 使用平均池化构建多尺度特征图列表
        feats = []
        for level in range(self.num_levels):
            if level == 0:
                feat = bev_proj
            else:
                scale = 2 ** level
                feat = F.avg_pool2d(bev_proj, kernel_size=scale, stride=scale)
            feats.append(feat)
            
        return feats

    def _sample_single_level(self, value_level, sampling_grid):
        """在单个尺度上执行高效的双线性插值采样 (grid_sample)

        输入:
            value_level:   [B, C, H_l, W_l]  当前尺度的 BEV 特征图
            sampling_grid: [B, Nq, M, K, 2]  映射到 [-1, 1] 区间的采样网格坐标

        输出:
            sampled:       [B, Nq, M, K, C_head]  当前尺度下采样到的特征
        """
        b, c, h_l, w_l = value_level.shape
        n_q = sampling_grid.shape[1]

        # -----------------------------------------------------------------
        # 1. 重构 Value 特征图，将 Batch 和 Head 维度合并，迎合 grid_sample 的要求
        # -----------------------------------------------------------------
        # [B, C, H_l, W_l] -> [B, M, C_head, H_l, W_l]
        value_reshaped = value_level.view(b, self.num_heads, self.head_dim, h_l, w_l)
        
        # [B, M, C_head, H_l, W_l] -> [B * M, C_head, H_l, W_l]
        value_reshaped = value_reshaped.view(b * self.num_heads, self.head_dim, h_l, w_l)

        # -----------------------------------------------------------------
        # 2. 重构 Grid 采样坐标
        # -----------------------------------------------------------------
        # [B, Nq, M, K, 2] -> [B, M, Nq, K, 2]
        grid = sampling_grid.permute(0, 2, 1, 3, 4).contiguous()
        
        # 将 Nq 和 K 组合看作输出网格的高和宽 H_out=Nq, W_out=K
        # [B, M, Nq, K, 2] -> [B * M, Nq, K, 2]
        grid = grid.view(b * self.num_heads, n_q, self.num_points, 2)

        # -----------------------------------------------------------------
        # 3. 核心 PyTorch 采样算子: F.grid_sample
        # 输入:  Input=[B*M, C_head, H_l, W_l], Grid=[B*M, Nq, K, 2]
        # 输出:  Sampled=[B*M, C_head, Nq, K]
        # -----------------------------------------------------------------
        sampled = F.grid_sample(
            value_reshaped,
            grid,
            mode='bilinear',       # 双线性插值
            padding_mode='zeros',  # 超出边界补 0
            align_corners=False
        )

        # -----------------------------------------------------------------
        # 4. 还原特征维度，分离 Batch、Head 与 Query
        # -----------------------------------------------------------------
        # [B * M, C_head, Nq, K] -> [B, M, C_head, Nq, K]
        sampled = sampled.view(b, self.num_heads, self.head_dim, n_q, self.num_points)
        
        # [B, M, C_head, Nq, K] -> [B, Nq, M, K, C_head]
        sampled = sampled.permute(0, 3, 1, 4, 2).contiguous()
        
        return sampled

    def forward(self, query, bev_feature):
        """前向传播函数

        输入:
            query:       [B, Nq, C]      目标/运动特征 Token
            bev_feature: [B, C, H, W]    2D BEV 鸟瞰特征图

        输出:
            fused_bev_feature: [B, C, H, W]  融入 Query 上下文后的 BEV 特征图
        """
        # -----------------------------------------------------------------
        # 0. 输入维度检查与基础变量提取
        # -----------------------------------------------------------------
        b, n_q, c = query.shape
        _, c_bev, h, w = bev_feature.shape
        if c != c_bev:
            raise ValueError(f"Query 通道数 ({c}) 与 BEV 通道数 ({c_bev}) 不匹配！")

        # -----------------------------------------------------------------
        # 1. 预先构建多尺度特征金字塔 (Value 已经在内部完成投影)
        # multi_scale_feats 是一个包含 L 个元素的列表:
        # Each feat_l: [B, C, H_l, W_l]
        # -----------------------------------------------------------------
        multi_scale_feats = self._build_multi_scale_features(bev_feature)

        # -----------------------------------------------------------------
        # 2. 从 Query 中预测 Reference Points (参考中心点)
        # -----------------------------------------------------------------
        # [B, Nq, C] -> [B, Nq, L * 2]
        ref_pts = self.reference_points_proj(query)
        # 用 Sigmoid 将坐标严格归一化到 [0, 1] 相对比例区间
        ref_pts = torch.sigmoid(ref_pts)
        # [B, Nq, L * 2] -> [B, Nq, L, 2]  (最后一维 2 代表 x, y 相对坐标)
        reference_points = ref_pts.view(b, n_q, self.num_levels, 2)

        # -----------------------------------------------------------------
        # 3. 从 Query 中预测 Offsets (采样点偏移量)
        # -----------------------------------------------------------------
        # [B, Nq, C] -> [B, Nq, M * L * K * 2]
        offsets = self.offset_proj(query)
        # [B, Nq, M * L * K * 2] -> [B, Nq, M, L, K, 2]
        sampling_offsets = offsets.view(b, n_q, self.num_heads, self.num_levels, self.num_points, 2)

        # -----------------------------------------------------------------
        # 4. 从 Query 中预测 Attention Weights (注意力权重)
        # -----------------------------------------------------------------
        # [B, Nq, C] -> [B, Nq, M * L * K]
        weights = self.weight_proj(query)
        # [B, Nq, M * L * K] -> [B, Nq, M, L * K]
        weights = weights.view(b, n_q, self.num_heads, self.num_levels * self.num_points)
        # 在 (L * K) 维度上归一化，即同一个 Head 内所有尺度所有采样点的权重和为 1
        attention_weights = F.softmax(weights, dim=-1)
        # [B, Nq, M, L * K] -> [B, Nq, M, L, K]
        attention_weights = attention_weights.view(b, n_q, self.num_heads, self.num_levels, self.num_points)

        # -----------------------------------------------------------------
        # 5. 多尺度可变形采样与特征加权聚合
        # -----------------------------------------------------------------
        # 初始化加权累加容器: [B, Nq, M, C_head]
        aggregated_context = query.new_zeros((b, n_q, self.num_heads, self.head_dim))

        for level, feat_l in enumerate(multi_scale_feats):
            # 获取当前尺度的特征图宽高
            _, _, h_l, w_l = feat_l.shape

            # 构建当前尺度的尺寸归一化常数: [1, 1, 1, 1, 2]
            normalizer = query.new_tensor([w_l, h_l]).view(1, 1, 1, 1, 2)

            # A. 计算当前尺度的相对偏移量 (绝对像素偏移 / 图像宽高)
            # [B, Nq, M, K, 2]
            offsets_l = sampling_offsets[:, :, :, level, :, :] / normalizer

            # B. 提取当前尺度的参考中心点: [B, Nq, 1, 1, 2]
            ref_l = reference_points[:, :, None, level, None, :]

            # C. 绝对物理相对坐标 (0~1 之间): [B, Nq, M, K, 2]
            locations_l = ref_l + offsets_l

            # D. 转换为 PyTorch grid_sample 要求的 [-1, 1] 坐标空间
            # 映射公式: grid = location * 2.0 - 1.0 (其中 0 -> -1, 1 -> +1)
            grid_l = locations_l * 2.0 - 1.0

            # E. 在当前尺度特征图上采样: [B, Nq, M, K, C_head]
            sampled_l = self._sample_single_level(feat_l, grid_l)

            # F. 提取当前尺度的权重并扩维: [B, Nq, M, K, 1]
            weights_l = attention_weights[:, :, :, level, :].unsqueeze(-1)

            # G. 加权求和 (对 K 个采样点求和) 并累加到总容器中
            # (sampled_l * weights_l) -> [B, Nq, M, K, C_head]
            # .sum(dim=3)             -> [B, Nq, M, C_head]
            aggregated_context = aggregated_context + (sampled_l * weights_l).sum(dim=3)

        # -----------------------------------------------------------------
        # 6. 多头特征拼接与 Query 更新
        # -----------------------------------------------------------------
        # [B, Nq, M, C_head] -> [B, Nq, M * C_head] = [B, Nq, C]
        query_update = aggregated_context.view(b, n_q, self.embed_dim)
        
        # 经过输出投影
        query_update = self.output_proj(query_update)
        
        # 残差连接与 LayerNorm
        updated_query = self.norm(query + self.dropout(query_update))

        # -----------------------------------------------------------------
        # 7. 将 Query 的全局高阶语义回注入 BEV 特征图
        # -----------------------------------------------------------------
        # 对所有 Queries 提取全局均值上下文: [B, Nq, C] -> [B, 1, C]
        global_query_context = updated_query.mean(dim=1, keepdim=True)
        
        # 非线性特征变换增强表达能力: [B, 1, C] -> [B, 1, C]
        global_query_context = self.context_fusion(global_query_context)

        # 展平 BEV 特征做广播相加
        # [B, C, H, W] -> [B, H*W, C]
        bev_tokens = bev_feature.flatten(2).transpose(1, 2)
        
        # 广播加法: [B, H*W, C] + [B, 1, C] -> [B, H*W, C]
        fused_tokens = bev_tokens + global_query_context

        # 还原为 2D 特征图形状: [B, H*W, C] -> [B, C, H*W] -> [B, C, H, W]
        fused_bev_feature = fused_tokens.transpose(1, 2).view(b, c, h, w).contiguous()

        return fused_bev_feature

In [7]:
# 假设 Batch Size 为 2，有 100 个 Query，通道数为 256
# BEV 特征图尺寸为 200x200
B, Nq, C = 2, 100, 256
H, W = 200, 200

# 伪造输入数据
dummy_query = torch.randn(B, Nq, C)
dummy_bev = torch.randn(B, C, H, W)

# 实例化融合模块
fusion_module = DeformableBEVFusion(
    embed_dim=256,
    num_heads=8,
    num_levels=3,
    num_points=4,
    dropout=0.1
)

# 前向计算
out_bev = fusion_module(dummy_query, dummy_bev)

print("输入 Query 形状:", dummy_query.shape)
print("输入 BEV 形状:  ", dummy_bev.shape)
print("输出 BEV 形状:  ", out_bev.shape)

assert out_bev.shape == dummy_bev.shape, "维度不一致，测试失败！"

输入 Query 形状: torch.Size([2, 100, 256])
输入 BEV 形状:   torch.Size([2, 256, 200, 200])
输出 BEV 形状:   torch.Size([2, 256, 200, 200])
